# NeMo MSDD vs Pyannote 3.1 — Diarization Comparison

Runs the same audio through both diarizers using a shared Whisper transcription (large-v3).
Compares accuracy by measuring speaker assignment agreement and optional YouTube caption alignment.

**Requirements:**
- Colab Pro (A100 recommended)
- HuggingFace token (accept Pyannote 3.1 license at https://huggingface.co/pyannote/speaker-diarization-3.1)

In [ ]:
# Version: v19 (2026-03-28)
print('Notebook v19 — 2026-03-28')

## 1. Setup

In [ ]:
# Clone the repo (force fresh clone)
%cd /content
!rm -rf /content/vault
!git clone --branch feat/people-agents https://github.com/cha7ura/vault.git /content/vault
%cd /content/vault/vendor/whisper-diarization

In [ ]:
# Install dependencies (runtime will restart after)
!apt-get install -y -qq nodejs > /dev/null 2>&1
!pip install -q "numpy<2"
!pip install -q "faster-whisper>=1.1.0"
!pip install -q "nemo-toolkit[asr]>=2.5.0"
!pip install -q "pyannote.audio>=3.1.0"
!pip install -q git+https://github.com/MahmoudAshraf97/demucs.git
!pip install -q git+https://github.com/oliverguhr/deepmultilingualpunctuation.git
!pip install -q yt-dlp nltk

# Restart runtime to pick up numpy<2
import os
os._exit(0)

In [ ]:
# Set your HuggingFace token (required for Pyannote 3.1)
import os
from google.colab import userdata

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab Secrets")
except Exception:
    os.environ["HF_TOKEN"] = ""  # <-- paste your token here if not using Secrets
    if not os.environ["HF_TOKEN"]:
        print("WARNING: HF_TOKEN not set. Pyannote will fail.")
    else:
        print("HF_TOKEN set manually")

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Input Audio

Paste a YouTube URL and it will be downloaded automatically.

In [ ]:
# Paste your YouTube URL here
YOUTUBE_URL = "https://www.youtube.com/watch?v=Cn8HBj8QAbk"

import subprocess, re
import os

vid_match = re.search(r'(?:v=|/)([a-zA-Z0-9_-]{11})', YOUTUBE_URL)
video_id = vid_match.group(1) if vid_match else 'audio'
audio_path = f'/content/{video_id}.wav'

if not os.path.exists(audio_path):
    %cd /content
    !yt-dlp -x --audio-format wav --postprocessor-args "ffmpeg:-ar 16000 -ac 1" -o "{video_id}.%(ext)s" "{YOUTUBE_URL}"
else:
    print(f'Audio already downloaded: {audio_path}')

print(f"\nAudio: {audio_path}")

## 3. Run Comparison Pipeline

In [ ]:
# ============================================================
# SETTINGS
# ============================================================
whisper_model = "large-v3"
batch_size = 16
device = "cuda"

# Demucs mode: "with" | "without" | "both"
demucs_mode = "both"

# Whisper no-speech threshold (lower = less likely to skip quiet speech)
# Default 0.6 — try 0.3 or 0.1 if Whisper skips sections
no_speech_threshold = 0.3

print(f"Whisper: {whisper_model}, Demucs: {demucs_mode}, no_speech_thresh: {no_speech_threshold}")

### Phase A: Whisper Transcription (shared, run once)

In [ ]:
import os, sys, time, json

import nltk
nltk.download('punkt_tab', quiet=True)

WHISPER_DIR = "/content/vault/vendor/whisper-diarization"
os.chdir(WHISPER_DIR)
if WHISPER_DIR not in sys.path:
    sys.path.insert(0, WHISPER_DIR)

from diarize import (
    run_whisper_transcription,
    run_diarization,
    run_postprocessing,
    save_whisper_cache,
)
print('Imports OK')

whisper_results = {}  # key: 'raw' or 'demucs'

# --- Without Demucs (raw audio) ---
if demucs_mode in ('without', 'both'):
    print(f"\n{'='*60}")
    print('WHISPER PASS 1: Raw audio (no Demucs)')
    print(f"{'='*60}")
    t0 = time.time()
    whisper_results['raw'] = run_whisper_transcription(
        audio_path, whisper_model, device, batch_size, 'en',
        no_speech_threshold=no_speech_threshold,
    )
    print(f"Done in {time.time()-t0:.1f}s — {len(whisper_results['raw']['word_timestamps'])} words")

# --- With Demucs (separated vocals) ---
if demucs_mode in ('with', 'both'):
    demucs_vocals = f'/content/demucs_out/htdemucs/{video_id}/vocals.wav'
    if not os.path.exists(demucs_vocals):
        print(f"\n{'='*60}")
        print('DEMUCS: Separating vocals...')
        print(f"{'='*60}")
        !python -m demucs.separate -n htdemucs --two-stems=vocals "{audio_path}" -o "/content/demucs_out" --device cuda
    else:
        print(f'\nReusing existing Demucs output: {demucs_vocals}')

    if os.path.exists(demucs_vocals):
        print(f"\n{'='*60}")
        print('WHISPER PASS 2: Demucs vocals')
        print(f"{'='*60}")
        t0 = time.time()
        whisper_results['demucs'] = run_whisper_transcription(
            demucs_vocals, whisper_model, device, batch_size, 'en',
            no_speech_threshold=no_speech_threshold,
        )
        print(f"Done in {time.time()-t0:.1f}s — {len(whisper_results['demucs']['word_timestamps'])} words")
    else:
        print('Demucs failed — skipping demucs pass')

print(f"\nWhisper passes completed: {list(whisper_results.keys())}")

### Phase B: Run All Diarizer Combinations

Runs each diarizer on each Whisper pass (raw / demucs).

In [ ]:
all_results = {}  # key: label, value: {speaker_ts, segments, diarize_time, post_time}

for audio_key, wr in whisper_results.items():
    suffix = '-demucs' if audio_key == 'demucs' else ''

    for diarizer_name in ['msdd', 'pyannote']:
        label = f"{'nemo-msdd' if diarizer_name == 'msdd' else 'pyannote-3.1'}{suffix}"
        print(f"\n{'='*60}")
        print(f"Running: {label}")
        print(f"{'='*60}")

        t0 = time.time()
        speaker_ts = run_diarization(wr['audio_waveform'], diarizer_name, device)
        diarize_time = time.time() - t0

        t0 = time.time()
        segments = run_postprocessing(
            wr['word_timestamps'], speaker_ts, wr['language'], audio_path,
        )
        post_time = time.time() - t0

        speakers = {s['speaker'] for s in segments}
        all_results[label] = {
            'speaker_ts': speaker_ts,
            'segments': segments,
            'diarize_time': diarize_time,
            'post_time': post_time,
            'n_speakers': len(speakers),
            'n_segments': len(segments),
            'n_turns': len(speaker_ts),
        }
        print(f"  {label}: {len(speakers)} speakers, {len(segments)} segments, {diarize_time:.1f}s")

print(f"\nCompleted: {list(all_results.keys())}")

## 4. Comparison Summary

In [ ]:
print(f"{'='*90}")
print('COMPARISON SUMMARY')
print(f"{'='*90}")
print(f"Audio: {audio_path}")
print(f"Whisper: {whisper_model}")
print(f"Demucs mode: {demucs_mode}")
print()

labels = list(all_results.keys())
col_w = 18
header = f"{'Metric':<25}" + ''.join(f"{l:>{col_w}}" for l in labels)
print(header)
print('-' * len(header))

for metric, key in [('Diarize time', 'diarize_time'), ('Post-process time', 'post_time'),
                     ('Speakers', 'n_speakers'), ('Speaker turns', 'n_turns'), ('Segments', 'n_segments')]:
    vals = []
    for l in labels:
        v = all_results[l][key]
        vals.append(f"{v:>{col_w-1}.1f}s" if 'time' in key else f"{v:>{col_w}}")
    print(f"{metric:<25}" + ''.join(vals))

# GPU memory usage
import torch
if torch.cuda.is_available():
    max_mem = torch.cuda.max_memory_allocated() / 1e9
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\nGPU: Peak {max_mem:.2f} GB / {total_mem:.1f} GB ({max_mem/total_mem*100:.1f}%)")

import psutil
ram = psutil.virtual_memory()
print(f"RAM: {ram.used/1e9:.1f} GB / {ram.total/1e9:.1f} GB ({ram.percent}%)")

## 6. Side-by-Side Transcript

In [ ]:
for label, res in all_results.items():
    print(f"\n--- {label} (first 10 segments) ---")
    for seg in res['segments'][:10]:
        print(f"  [{seg['start']:7.1f}s - {seg['end']:7.1f}s] {seg['speaker']}: {seg['text'][:70]}")

## 8. Upload to Supabase

Upload both diarizer results to your Supabase `segments` table for side-by-side comparison in the vault frontend.
Add `SUPABASE_URL` and `SUPABASE_SERVICE_ROLE_KEY` (service role) to Colab Secrets.

In [ ]:
from google.colab import userdata
import requests

SUPABASE_URL = userdata.get("SUPABASE_URL")
SUPABASE_SERVICE_ROLE_KEY = userdata.get("SUPABASE_SERVICE_ROLE_KEY")

headers = {
    "apikey": SUPABASE_SERVICE_ROLE_KEY,
    "Authorization": f"Bearer {SUPABASE_SERVICE_ROLE_KEY}",
    "Content-Type": "application/json",
    "Prefer": "return=representation",
}

# Get or create episode
res = requests.get(
    f"{SUPABASE_URL}/rest/v1/episodes?youtube_id=eq.{video_id}&select=id",
    headers=headers,
)
episodes = res.json()

if episodes:
    episode_id = episodes[0]["id"]
    print(f"Found existing episode: {episode_id}")
else:
    res = requests.post(
        f"{SUPABASE_URL}/rest/v1/episodes",
        headers=headers,
        json={"youtube_id": video_id, "title": f"Episode {video_id}",
              "duration_seconds": int(list(whisper_results.values())[0]['audio_duration'])},
    )
    if res.status_code >= 300:
        print(f"Error creating episode: {res.text}")
    else:
        episode_id = res.json()[0]["id"]
        print(f"Created episode: {episode_id}")

# Delete existing comparison segments
diarizer_labels = ','.join(all_results.keys())
requests.delete(
    f"{SUPABASE_URL}/rest/v1/segments?episode_id=eq.{episode_id}&diarizer=in.({diarizer_labels})",
    headers=headers,
)

def upload_segments(segments_data, diarizer_name):
    rows = [{
        "episode_id": episode_id,
        "start_time": seg["start"], "end_time": seg["end"],
        "text": seg["text"], "speaker": seg["speaker"],
        "words": json.dumps(seg.get("words", [])),
        "diarizer": diarizer_name,
    } for seg in segments_data]
    for b in range(0, len(rows), 100):
        res = requests.post(
            f"{SUPABASE_URL}/rest/v1/segments",
            headers={**headers, "Prefer": "return=minimal"},
            json=rows[b:b+100],
        )
        if res.status_code >= 300:
            print(f"Error: {res.text}")
            return
    print(f"  Uploaded {len(rows)} segments for {diarizer_name}")

for label, res in all_results.items():
    upload_segments(res['segments'], label)

print(f"\nDone! All {len(all_results)} diarizer results uploaded.")

## 7. Save & Download Results

In [ ]:
# Save all results
base = audio_path.rsplit('.', 1)[0]

for label, res in all_results.items():
    path = f"{base}_{label.replace('.', '')}.json"
    with open(path, 'w') as f:
        json.dump(res['segments'], f, ensure_ascii=False, indent=2)
    print(f"Saved: {path}")

# Benchmark summary
benchmark = {
    'video_id': video_id,
    'whisper_model': whisper_model,
    'demucs_mode': demucs_mode,
    'results': {l: {k: v for k, v in r.items() if k != 'segments' and k != 'speaker_ts'}
               for l, r in all_results.items()},
}
with open(f"{base}_benchmark.json", 'w') as f:
    json.dump(benchmark, f, ensure_ascii=False, indent=2)
print(f"Saved: {base}_benchmark.json")

In [ ]:
from google.colab import files

base = audio_path.rsplit('.', 1)[0]
for label in list(all_results.keys()) + ['benchmark']:
    path = f"{base}_{label.replace('.', '')}.json"
    if os.path.exists(path):
        files.download(path)